In [ ]:
import pandas as pd 
from hr_data_analysis.prep import clean_hr_data

df = pd.read_parquet('../data/processed/hr_data.parquet')

df.head(20)



### Company Profile
Q1, 3, 4, 8 11

#### Q1: How Many Employees in each Department

In [ ]:
df['Department'].value_counts().sort_values(ascending=False)

Finance is the largest department

Q3: Gender Distribution

In [ ]:
df['Gender'].value_counts()

Almost equal number of male and female employees 

Q4: Average Employee Age

In [ ]:
print(df['Age'].mean())

Averge age of the employees is 36 years

Q8: Most common Positions held

In [ ]:
df['Position'].value_counts().sort_values(ascending=False)

Assistants are the most common 

Q11: Percentage of employees enrolled in benefits

In [ ]:
enrolled_in_benefits = df[df['Benefits Enrolled'] == 'Yes'].shape[0] 

total_employees = df.shape[0]

pct_enrolled = (enrolled_in_benefits / total_employees) * 100 

pct_enrolled

42.9% of employees are enrolled in benefits

### Pay: 
Q2, 14, 16, 17 (averages and group comparisons)


Q2: What is the average salary in the company? 

In [ ]:
print(df['Salary'].mean())

Average salary is 60207.701

Q14: Which department has the highest salaries?

In [ ]:
df.groupby('Department')['Salary'].agg('mean').round(2).sort_values(ascending=False)

Sales is the highest in terms of salary 

Q16: Do high performers earn more? 

In [ ]:
df.groupby('Performance Score')['Salary'].agg('mean').round(2).sort_values(ascending=False)

Not necessarily, high performers do not earn more than the others

Q17. What is the male vs female salary difference? 

In [ ]:
gendered_salary = df.groupby('Gender')['Salary'].agg('mean').round(2).sort_values(ascending=False)

print(gendered_salary)

gendered_salary_diff = gendered_salary['Female'] - gendered_salary['Male']

print(gendered_salary_diff)

Women earn higher in comparison to men but by a small margin thus almost equal pay
With a difference of 792.43

### Tenure, time off, and development: 
Q5, 9, 10, 13

Q5: How long do employees typically stay at the company? 

In [ ]:
print(df['Years of Service'].agg(['mean', 'median']))

On average 4.531 Years at the company
Employees stay at the company for a median of 5 years

Q9: How many sick days do employees take on average?

In [ ]:
print(df['Sick Days Taken'].agg('mean'))   

6.37 Days taken on average for sick days taken

Q10:  How much vacation time is unused?

In [ ]:
print(df['Vacation Days Remaining'].agg('mean') )

10 Days on average of Vacation Days are unused/remaining

Q13: How much training do employees get? 

In [ ]:
print(df['Training Hours'].agg('mean') )

41.81 hrs of average training hours 

### Satisfaction and attrition: 
Q6, 7, 15, 19, 20 (the core of the story)

Q6: What are the job satisfaction levels? 

In [ ]:
print(pd.concat([
    df['Job Satisfaction'].value_counts().sort_values(ascending=False),
    (df['Job Satisfaction'].value_counts(normalize=True)*100).sort_values(ascending=False)
], axis=1, keys=['Count', 'Percentage']).round(2))

Many unsatisfied employees based on the job satisfaction score
with levels 1 and 2 carrying major number between the while being neutral looks like they are satisfied

Q7: How many employees left the company? 

In [ ]:
df[df['Attrition'] == 'Yes'].shape[0]

392 people left the company
Making the attrition rate to be 39.2%

Q15:  Are satisfied employees less likely to leave? 

In [ ]:
df['Job Satisfaction'].value_counts().sort_values(ascending=False)
df['Attrition'].value_counts().sort_values(ascending=False)

# (pd.crosstab(
#     index = df['Job Satisfaction'], 
#     columns = df['Attrition'], 
#     normalize='index'
# )*100).round(2)

counts = pd.crosstab(df['Job Satisfaction'], df['Attrition'], margins=True)
pcts = (pd.crosstab(df['Job Satisfaction'], df['Attrition'], normalize='index') * 100).round(2)

print(pd.concat([df['Job Satisfaction'].value_counts(), pcts['Yes']], axis=1, keys=['counts', 'pcts']))


Yes satisfied people are less likely to leave since the higher the satisfaction the lower the rate of attrition 
Low satisfaction shows a higher leaving rate

Percentages alone can be misleading if one satisfaction level only has 3 people. Paired it with counts so you know the sample size is reliable:

In [ ]:
# 1. Get raw counts (with row totals)
# Take the 'All' column that contains the number of employees in each department
counts = pd.crosstab(df['Job Satisfaction'], df['Attrition'], margins=True)

# 2. Get attrition rate (% of each satisfaction level that answered 'Yes')
attrition_rate = (pd.crosstab(
    df['Job Satisfaction'], 
    df['Attrition'], 
    normalize='index'
)['Yes'] * 100).round(2)

# Combine into one clear summary table
summary = pd.DataFrame({
    'Total Employees': counts['All'].drop('All'),
    'Employees Left': counts['Yes'].drop('All'),
    'Attrition Rate (%)': attrition_rate
})

print(summary)


Q19:  Which department has the best job satisfaction? 

In [ ]:
dept_satisfaction = (
    df.groupby('Department')['Job Satisfaction']
    .agg(Count='count', Average_Satisfaction='mean')
    .round(2)
    .sort_values(by='Average_Satisfaction', ascending=False)
)

print(dept_satisfaction)


Finance has the lowest average satisfaction among all the departments at 2.72

Q20: Are Full-Time employees more satisfied than Part-Time? 

In [ ]:
employment_type_satisfaction = (df.groupby('Employment Status')['Job Satisfaction']
                                .agg(Count='count',Average_Satisfaction='mean')
                                .round(2)
                                .sort_values(by='Average_Satisfaction', ascending=False))

print(employment_type_satisfaction)

The 3 groups are nearly identical so it doesn't change peoples satisfaction of the job. So Full-Time Employees are on average almost the same satisfaction with the rest

### Performance: 
Q12, 16

Q12: What are the performance score distributions? 

In [ ]:
df['Performance Score'].value_counts().sort_values(ascending=False)

B is the most common Performance score 

Q16: Do high performers earn more? 

In [ ]:
performers_salary = (
    df.groupby('Performance Score')['Salary']
    .mean()
    .round(2)
    .sort_values(ascending=False)
    )

performers_salary

Theres no significant pattern in salary pay from performance score 